In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

In [ ]:
# - 10 juin / now
interval_start = '2026-06-10 00:00:00'
interval_stop = '2026-07-01 00:00:00'

In [ ]:
all_columns = [
    'action_id',
    'idsite',
    'idvisit',
    'actions',
    'country',
    'region',
    'city',
    'operatingsystemname',
    'devicemodel',
    'devicebrand',
    'visitduration',
    'dayssincefirstvisit',
    'visitortype',
    'sitename',
    'userid',
    'serverdateprettyfirstaction',
    'action_type',
    'action_eventcategory',
    'action_eventaction',
    'action_eventname',
    'action_eventvalue',
    'action_timespent',
    'action_timestamp',
    'usercustomproperties',
    'usercustomdimensions',
    'dimension1',
    'dimension2',
    'dimension3',
    'dimension4',
    'dimension5',
    'dimension6',
    'dimension7',
    'dimension8',
    'dimension9',
    'dimension10',
    'action_url',
    'sitesearchkeyword',
    'action_title',
    'visitorid',
    'referrertype',
    'referrername',
    'resolution',
    'experiments']

In [ ]:
columns = ['action_id',
    'idvisit',
    'actions',
    'operatingsystemname',
    'action_type',
    'action_eventcategory',
    'action_eventaction',
    'action_eventname',
    'action_eventvalue',
    'action_url',
    'experiments']

range_query = f"""
        SELECT {", ".join(columns)} FROM matomo_partitioned
        WHERE action_timestamp >= '{interval_start}'
          AND action_timestamp < '{interval_stop}'
        """

In [ ]:
filter_urls = lambda field : f"""(
              {field} ~ '^https://code\\.travail\\.gouv\\.fr/contribution/[a-zA-Z]+.*'
              OR
              {field} ~ '^https://code\\.travail\\.gouv\\.fr/outils/*'
              OR
              {field} ~ '^https://code\\.travail\\.gouv\\.fr/modeles-de-courriers/*'
              )"""

### we retrieve visits for contribs / outils / modeles de courrier

In [ ]:
query_visits =  range_query + f"""
          AND action_type = 'action' 
          AND {filter_urls('action_url')}
          ORDER BY action_timestamp asc;
    """

visits_data = await matomo.run_query(query_visits)

In [ ]:
visits_df = pd.DataFrame(visits_data, columns=columns)

### we retrieve events of interest to analyze contribs / outils / modeles de courrier

In [ ]:
event_names = ['start', 'results', 'result', 'indemnite']
event_actions = ['type_CTRL_C', 'click_afficher_les_informations_sans_CC', 'click_afficher_les_informations_générales', 'click_afficher_les_informations_CC', 'cc_select_traitée']

In [ ]:
# events required to analyse tools / models / contribs
query_events = range_query + f"""
          AND {filter_urls('action_url')}
          AND action_type = 'event' 
          AND (
              action_eventname IN ({", ".join([f"'{n}'" for n in event_names])})
              OR
              action_eventaction IN ({", ".join([f"'{a}'" for a in event_actions])})
              )
          ORDER BY action_timestamp asc;
    """

events_data = await matomo.run_query(query_events)
events_df = pd.DataFrame(events_data, columns=columns)

### we retrieve download events for modeles

In [ ]:
query_downloads = range_query + "AND action_type='download'"
downloads_data = await matomo.run_query(query_downloads)
downloads_df = pd.DataFrame(downloads_data,columns=columns)

# manipulate data in order the merge them all

In [ ]:
id_visit_col = 'id_visit'
content_type_col = 'content_type'
content_slug_col = 'content_slug'
event_type_col = 'event_type'

In [ ]:
i = len('https://code.travail.gouv.fr/')

def get_type_slug(urls):
    content_type = urls.str[i:].str.split('/').str[0]
    content_slug = urls.str[i:].str.split('/').str[1].str.split('?').str[0].str.split('#').str[0]
    # todo filter valid slugs ?

    return (content_type, content_slug)

def is_mobile(df):
   return df.operatingsystemname.apply(lambda x : x in ['iOS', 'Android'])

def get_xp(df):
    xp_columns = pd.DataFrame(df['experiments'].apply(eval).tolist())
    return pd.DataFrame({'xp': xp_columns[0].apply(lambda x : x['variation']['name'] if x is not None else None)})

In [ ]:
visit_content_type, visit_content_slug = get_type_slug(visits_df['action_url'])

visit_structured = pd.DataFrame({id_visit_col: visits_df['idvisit'], content_type_col: visit_content_type, content_slug_col: visit_content_slug})
visit_structured[event_type_col] = 'visit'
visit_structured['mobile'] = is_mobile(visits_df)
visit_structured['xp'] = get_xp(visits_df)

In [ ]:
visit_structured

In [ ]:
event_content_type, event_content_slug = get_type_slug(events_df['action_url'])
event_structured = pd.DataFrame({id_visit_col: events_df['idvisit'], content_type_col: event_content_type, content_slug_col: event_content_slug})
event_structured[event_type_col] = 'none'
event_structured['mobile'] = is_mobile(events_df)
event_structured['xp'] = get_xp(events_df)


In [ ]:
# events modeles
mask_ctrl_c = (events_df['action_eventaction'] == 'type_CTRL_C')
event_structured[event_type_col] = event_structured[event_type_col].mask(mask_ctrl_c, 'model_copy')

In [ ]:
# events tools
mask_start = events_df['action_eventname'].isin(['start', 'intro']) & (event_structured.content_type == 'outils')
mask_results = events_df['action_eventname'].isin(['results', 'result', 'indemnite']) & (event_structured.content_type == 'outils')

In [ ]:
event_structured[event_type_col] = event_structured[event_type_col].mask(mask_start, 'tool_start')
event_structured[event_type_col] = event_structured[event_type_col].mask(mask_results, 'tool_results')

In [ ]:
# events cc
mask_no_cc = events_df['action_eventaction'].isin(['click_afficher_les_informations_sans_CC']) & (event_structured.content_type == 'contribution')
mask_global = events_df['action_eventaction'].isin(['click_afficher_les_informations_générales']) & (event_structured.content_type == 'contribution')
mask_cc = events_df['action_eventaction'].isin(['click_afficher_les_informations_CC']) & (event_structured.content_type == 'contribution')

event_structured[event_type_col] = event_structured[event_type_col].mask(mask_no_cc, 'no_cc')
event_structured[event_type_col] = event_structured[event_type_col].mask(mask_cc, 'cc')
event_structured[event_type_col] = event_structured[event_type_col].mask(mask_global, 'global')

In [ ]:
event_structured[event_type_col].value_counts()

In [ ]:
mask_dl = visit_structured.id_visit.isin(downloads_df.idvisit) & (visit_structured.content_type == 'modeles-de-courriers')

In [ ]:
download_structured = visit_structured[mask_dl].copy()
download_structured[event_type_col] = 'download'

In [ ]:
structured_dedup = pd.concat([download_structured, visit_structured, event_structured], axis=0).drop_duplicates().sort_values(by=['id_visit'])

In [ ]:
structured_dedup

In [ ]:
mod_logs = structured_dedup[structured_dedup.content_type == 'modeles-de-courriers']

mod_results = {}

def compute_mod(mod_df):
    visits = mod_df[mod_df.event_type == 'visit'].shape[0]
    downloads = mod_df[mod_df.event_type == 'download'].shape[0]
    copies = mod_df[mod_df.event_type == 'model_copy'].shape[0]
    return {'visits': visits, 'downloads': downloads, 'copies': copies, 
            'ratio_download_visits': f"{downloads/visits if visits > 0 else 0:.0%}", 
            'ratio_copy_visits': f"{copies/visits if visits > 0 else 0:.0%}",
            'ratio_action_visits': f"{(copies+downloads)/visits if visits > 0 else 0:.0%}"}

for mod in mod_logs.content_slug.value_counts().index:
    mod_slice = mod_logs[mod_logs.content_slug == mod]
    if mod_slice[mod_slice.event_type == 'visit'].shape[0] > 1 :        
        mod_metrics = compute_mod(mod_slice)
        mod_results[mod] = mod_metrics

mod_results['all'] = compute_mod(mod_logs)

In [ ]:
pd.DataFrame.from_dict(mod_results, orient='index').sort_values(by=['visits'], ascending=False)

In [ ]:
tool_logs = structured_dedup[structured_dedup.content_type == 'outils']
tool_results = {}

def compute_tool(tool_df):
    visits = tool_df[tool_df.event_type == 'visit'].shape[0]
    starts = tool_df[tool_df.event_type == 'tool_start'].shape[0]
    results = tool_df[tool_df.event_type == 'tool_results'].shape[0]
    return {'visits': visits, 'start': starts, 'results': results, 
            'ratio_start_visits': f"{starts/visits if visits > 0 else 0:.0%}", 
            'ratio_results_start': f"{results/starts if starts > 0 else 0:.0%}"}

for tool in tool_logs.content_slug.value_counts()[:10].index:
    tool_metrics = compute_tool(tool_logs[tool_logs.content_slug == tool])
    tool_results[tool] = tool_metrics

tool_results['all'] = compute_tool(tool_logs)

In [ ]:
pd.DataFrame.from_dict(tool_results, orient='index').sort_values(by=['visits'], ascending=False)

In [ ]:
structured_dedup.xp.value_counts()

In [ ]:
contrib_logs = structured_dedup[(structured_dedup.content_type == 'contribution') & (structured_dedup.xp == 'selon-ma-cc')]

In [ ]:
contrib_logs.event_type.value_counts()

In [ ]:
contribs_results = {}

def compute_contrib(contrib_df):
    [visits, cc, generic, no_cc] = [contrib_df[contrib_df.event_type == et].shape[0] for et in ['visit', 'cc', 'global', 'no_cc']]

    actions = cc+generic+no_cc
        
    return {'visits': visits, 'select_cc': cc, 'no_cc': no_cc, 'generic': generic,
            'ratio_actions_visits': f"{(actions)/visits if visits > 0 else 0:.0%}", 
            'ratio_cc_actions': f"{cc/actions if actions > 0 else 0:.0%}", 
            'ratio_generic_actions': f"{generic/actions if actions > 0 else 0:.0%}",
            'ratio_no_cc_actions': f"{no_cc/actions if actions > 0 else 0:.0%}"}

for contrib in contrib_logs.content_slug.value_counts().index:
    contrib_metrics = compute_contrib(contrib_logs[contrib_logs.content_slug == contrib])
    contribs_results[contrib] = contrib_metrics

contribs_results['all'] = compute_contrib(contrib_logs)

In [ ]:
pd.DataFrame.from_dict(contribs_results, orient='index').sort_values(by=['visits'], ascending=False)